# Boxplot Visualization Milestone

## Learning Objectives
- Understand what a boxplot represents
- Create boxplots for single and multiple columns
- Interpret median, quartiles, IQR, and outliers
- Compare distributions across columns
- Use boxplots as part of EDA

## Section 1: Understanding Boxplots

A **boxplot** (also called box-and-whisker plot) shows:
1. **Minimum** - lower whisker end
2. **Q1** (First Quartile) - 25th percentile (bottom of box)
3. **Median** (Q2) - 50th percentile (line in box)
4. **Q3** (Third Quartile) - 75th percentile (top of box)
5. **Maximum** - upper whisker end
6. **Outliers** - individual points beyond whiskers

### Key Concepts:
- **IQR** = Q3 - Q1 (height of the box)
- **Whiskers** extend to 1.5 × IQR from Q1 and Q3
- **Wider boxes** = more variability
- **Points beyond whiskers** = potential outliers

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configure matplotlib for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Section 2: Load and Prepare Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/tours.csv')

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"\nColumns: {df.columns.tolist()}")

# Display first few rows
df.head(10)

In [ ]:
# Convert numeric columns, handling errors
df['Visitors'] = pd.to_numeric(df['Visitors'], errors='coerce')
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Show basic statistics
df[['Visitors', 'Revenue', 'Rating']].describe()

## Section 3: Creating a Boxplot for a Single Column

Start by visualizing one column's distribution.

In [ ]:
# Create a boxplot for Visitors column
fig, ax = plt.subplots(figsize=(8, 6))

# Create boxplot
bp = ax.boxplot(df['Visitors'].dropna(), 
                vert=True,  # Vertical orientation
                patch_artist=True,  # Fill with color
                showmeans=True,  # Show mean as a diamond
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

# Customize appearance
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)

ax.set_ylabel('Number of Visitors', fontsize=12)
ax.set_title('Distribution of Tour Visitors', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.show()

# Calculate key statistics
visitors_data = df['Visitors'].dropna()
print(f"\nKey Statistics:")
print(f"Median: {visitors_data.median():.0f}")
print(f"Q1 (25%): {visitors_data.quantile(0.25):.0f}")
print(f"Q3 (75%): {visitors_data.quantile(0.75):.0f}")
print(f"IQR: {visitors_data.quantile(0.75) - visitors_data.quantile(0.25):.0f}")

### 💡 Interpretation
- The **line in the middle** of the box is the median
- **50% of the data** falls within the box (Q1 to Q3)
- The **whiskers** show the range of most data
- Any **points outside** whiskers are potential outliers

## Section 4: Comparing Boxplots Across Multiple Columns

Compare distributions side by side.

In [ ]:
# Method 1: Separate subplots for each column
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

columns = ['Visitors', 'Revenue', 'Rating']
colors = ['steelblue', 'coral', 'lightgreen']

for idx, (col, color) in enumerate(zip(columns, colors)):
    data = df[col].dropna()
    bp = axes[idx].boxplot(data, 
                           vert=True,
                           patch_artist=True,
                           showmeans=True,
                           meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
    
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.7)
    
    axes[idx].set_ylabel(col, fontsize=12)
    axes[idx].set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add median value
    median_val = data.median()
    axes[idx].text(0.5, 0.95, f'Median: {median_val:.1f}',
                   transform=axes[idx].transAxes,
                   fontsize=10, ha='center', va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

fig.suptitle('Comparing Distributions Across Columns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: All columns on one plot (normalized)
from sklearn.preprocessing import StandardScaler

# Normalize the data for fair comparison
scaler = StandardScaler()
df_normalized = df[['Visitors', 'Revenue', 'Rating']].copy()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df_normalized.dropna()),
    columns=['Visitors', 'Revenue', 'Rating']
)

# Create comparison boxplot
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([df_normalized['Visitors'], 
                  df_normalized['Revenue'], 
                  df_normalized['Rating']],
                labels=['Visitors', 'Revenue', 'Rating'],
                patch_artist=True,
                showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

# Color each box
colors = ['steelblue', 'coral', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Standardized Values (Z-scores)', fontsize=12)
ax.set_title('Normalized Distribution Comparison\n(All on same scale)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Mean')
ax.legend()

plt.tight_layout()
plt.show()

### 💡 Comparison Insights
- Compare which columns have **wider boxes** (more variability)
- Check which have **more outliers**
- Notice differences in **symmetry** (median position in box)
- Normalized plots allow **fair side-by-side comparison**

## Section 5: Identifying and Interpreting Outliers

Outliers are not always errors!

In [ ]:
# Revenue boxplot with outlier detection
revenue_data = df['Revenue'].dropna()
q1 = revenue_data.quantile(0.25)
q3 = revenue_data.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Identify outliers
outliers = revenue_data[(revenue_data < lower_bound) | (revenue_data > upper_bound)]

# Create boxplot
fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(revenue_data, 
                vert=True,
                patch_artist=True,
                showmeans=True,
                showfliers=True,  # Show outliers
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8),
                flierprops=dict(marker='o', markerfacecolor='red', 
                               markersize=10, alpha=0.7))

bp['boxes'][0].set_facecolor('coral')
bp['boxes'][0].set_alpha(0.7)

# Add outlier bounds
ax.axhline(y=lower_bound, color='green', linestyle='--', alpha=0.5, 
           label=f'Lower bound: ${lower_bound:.0f}')
ax.axhline(y=upper_bound, color='orange', linestyle='--', alpha=0.5,
           label=f'Upper bound: ${upper_bound:.0f}')

ax.set_ylabel('Revenue ($)', fontsize=12)
ax.set_title('Revenue Distribution with Outlier Bounds', 
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.legend()

plt.show()

print(f"\nOutlier Analysis:")
print(f"Number of outliers: {len(outliers)}")
if len(outliers) > 0:
    print(f"Outlier values: {outliers.values}")
    print(f"Percentage of data: {len(outliers)/len(revenue_data)*100:.1f}%")
else:
    print("No outliers detected")

### 💡 Understanding Outliers
- **Outlier definition**: Points more than 1.5 × IQR from Q1 or Q3
- **Not always mistakes**: Could be exceptional tours
- **Investigate first**: Check if they're data errors or legitimate values
- **Context matters**: A high-revenue tour might be a successful premium package

In [ ]:
# Let's look at which tours are outliers
if len(outliers) > 0:
    outlier_tours = df[df['Revenue'].isin(outliers)]
    print("Tours identified as outliers:")
    display(outlier_tours[['TourID', 'Location', 'Visitors', 'Revenue', 'Rating']])
else:
    print("No outlier tours to display")

## Section 6: Horizontal Boxplots for Better Labels

In [ ]:
# Horizontal boxplot
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [df['Rating'].dropna(), 
                df['Visitors'].dropna(), 
                df['Revenue'].dropna()]
labels = ['Rating (1-5)', 'Visitors (count)', 'Revenue ($)']

bp = ax.boxplot(data_to_plot,
                labels=labels,
                vert=False,  # Horizontal!
                patch_artist=True,
                showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

colors = ['lightgreen', 'steelblue', 'coral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Values', fontsize=12)
ax.set_title('Horizontal Boxplot Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## Section 7: Summary

### ✅ What You've Learned

1. **Boxplot Components**
   - Q1, Median (Q2), Q3 form the box
   - IQR = Q3 - Q1 (box height)
   - Whiskers extend to 1.5 × IQR
   - Outliers appear as individual points

2. **Creating Boxplots**
   - Single column: Shows one distribution
   - Multiple columns: Compare side-by-side
   - Normalized: Fair comparison across scales

3. **Interpreting Results**
   - Median shows central tendency
   - Box width shows variability (IQR)
   - Asymmetric box suggests skewness
   - Outliers need investigation, not automatic removal

4. **When to Use Boxplots**
   - Comparing distributions across groups
   - Quick outlier detection
   - Understanding spread and quartiles
   - Complementing histograms

### 🎯 Next Steps
- Combine boxplots with histograms in your EDA
- Use boxplots to compare across categorical groups
- Always investigate outliers before deciding actions
- Practice interpreting box asymmetry (skewness)

## 🎓 Practice Exercise

Try these on your own:
1. Create a boxplot for another dataset column
2. Calculate IQR manually and compare to plot
3. Find outliers programmatically
4. Create a boxplot grouped by a categorical variable (advanced)